# Fine-Tuning LLaMA 3.2 3B com LoRA — Assistente de Diabetes **v5**

**Objetivo:** Treinar um modelo para interpretar saídas de modelos de ML de diagnóstico de diabetes,
gerando explicações humanizadas para médicos com base nas diretrizes da ADA.

**Stack:** Transformers + PEFT (LoRA) + TRL (SFTTrainer) + MLflow  
**Ambiente:** Google Colab — coloque `train_v5.jsonl`, `val_v5.jsonl` e `test_v5.jsonl` na **raiz do Drive** antes de rodar.

---
| # | Correção aplicada |
|---|---|
| 1 | Adaptado para Google Colab + Google Drive |
| 2 | `trainer.evaluate()` chamado antes de `save_model` (evita RuntimeError do NotebookProgressCallback) |
| 3 | `mlflow.end_run()` antes de `start_run` em células subsequentes |
| 4 | `model_inf` recarregado via `PeftModel` antes da avaliação holdout |
| 5 | HF Token via `userdata.get` (Colab Secrets) com fallback manual |
| 6 | Todos os caminhos apontam para `/content/drive/MyDrive/` |


## 🌐 Setup do Servidor MLflow (Máquina Local)

> Execute estes comandos **no terminal WSL** antes de rodar o notebook.

### 1. Subir o MLflow com Docker
```bash
docker start mlflow-server 2>/dev/null || docker run -d \
  --name mlflow-server \
  --restart unless-stopped \
  -p 5000:5000 \
  -v $(pwd)/mlruns_salvo:/mlruns \
  -e MLFLOW_SERVER_CORS_ALLOWED_ORIGINS="*" \
  ghcr.io/mlflow/mlflow:latest \
  mlflow server \
  --backend-store-uri sqlite:////mlruns/mlflow.db \
  --default-artifact-root /mlruns \
  --host 0.0.0.0 \
  --port 5000
```

### 2. Configurar ngrok (apenas na primeira vez)
```bash
ngrok config add-authtoken SEU_TOKEN_AQUI
```
> 🔑 Obtenha seu token em: https://dashboard.ngrok.com/get-started/your-authtoken

### 3. Subir o ngrok e obter a URL
```bash
nohup ngrok http 5000 --host-header="localhost:5000" > ~/ngrok.log 2>&1 &
sleep 4
curl -s http://localhost:4040/api/tunnels | python3 -c "import sys,json; t=json.load(sys.stdin)['tunnels']; print(t[0]['public_url'])"
```

### 4. Atualizar a URL no notebook
Cole a URL retornada na variável `MLFLOW_URI` da célula de configuração abaixo.

> ⚠️ **Atenção:** A URL do ngrok muda a cada reinicialização. Repita o passo 3 e atualize `MLFLOW_URI` sempre que reiniciar.

## 1. Montar Google Drive e Instalar Dependências

In [ ]:
# Monte o Drive — os JSONLs devem estar na raiz do MyDrive
from google.colab import drive
drive.mount('/content/drive')

# Instala dependências (só precisa rodar uma vez por sessão)
%pip install -q --upgrade pip
%pip install -q transformers==4.47.1 peft==0.14.0 trl==0.13.0 \
              accelerate==1.2.1 bitsandbytes==0.45.0 datasets mlflow
print("✅ Dependências instaladas.")


## 2. Imports e Configuração de Paths

In [ ]:
import os, json, re, statistics, torch
from google.colab import userdata

# ── CONFIGURAÇÃO ─────────────────────────────────────────────────────
# HF Token: adicione em Colab > Secrets com o nome HF_TOKEN
HF_TOKEN = userdata.get('HF_TOKEN') or os.environ.get('HF_TOKEN', '')
MODEL_ID  = 'meta-llama/Llama-3.2-3B-Instruct'

# Todos os arquivos na raiz do Google Drive
BASE_DIR  = '/content/drive/MyDrive'

# ─────────────────────────────────────────────────────────────────────
# Devido limitação de tamanho de arquivos no GitHub, utilizar o
# caminho Google Drive que contém os arquivos de Treino, Validação e Teste
# https://drive.google.com/drive/folders/1LfLRFZG3ZRy_k6rD2mM-yMUmtOFaay0s?usp=sharing

TRAIN_JSONL = os.path.join(BASE_DIR, 'train_v5.jsonl')
VAL_JSONL   = os.path.join(BASE_DIR, 'val_v5.jsonl')
TEST_JSONL  = os.path.join(BASE_DIR, 'test_v5.jsonl')
LOCAL_OUT   = os.path.join(BASE_DIR, 'modelo_diabetes_v5')

os.makedirs(LOCAL_OUT, exist_ok=True)

print(f'BASE_DIR   : {BASE_DIR}')
print(f'train_v5   : {os.path.exists(TRAIN_JSONL)}')
print(f'val_v5     : {os.path.exists(VAL_JSONL)}')
print(f'test_v5    : {os.path.exists(TEST_JSONL)}')
print(f'HF_TOKEN   : {"✅ definido" if HF_TOKEN else "❌ necessário — adicione em Colab > Secrets"}')
print(f'CUDA       : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "❌ GPU não encontrada"}')


## 3. Configuração do MLflow

> **Colab:** MLflow fica disponível localmente. Use `ngrok` para expor a UI externamente se quiser,
> ou apenas confie nos logs salvos em disco. Se preferir pular o MLflow, o treino continua normalmente.


In [ ]:
import mlflow

# URI local — rode `mlflow ui` num terminal separado ou use ngrok
MLFLOW_URI      = os.environ.get('MLFLOW_URI', 'http://localhost:5000')
EXPERIMENT_NAME = 'Finetuning_Llama3.2_3B_LoRA_Diabetes_v5'

mlflow.set_tracking_uri(MLFLOW_URI)

try:
    mlflow.set_experiment(EXPERIMENT_NAME)
    MLFLOW_OK = True
    print(f'✅ MLflow conectado: {MLFLOW_URI}')
except Exception as e:
    MLFLOW_OK = False
    print(f'⚠️  MLflow indisponível ({e})')
    print('   Treinamento continua sem logging MLflow.')


## 4. Carregar o Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files={
    'train':      TRAIN_JSONL,
    'validation': VAL_JSONL,
})
test_dataset_raw = load_dataset('json', data_files={'test': TEST_JSONL})['test']

print(f'Treino:    {len(dataset["train"])} amostras')
print(f'Validação: {len(dataset["validation"])} amostras')
print(f'Teste:     {len(test_dataset_raw)} amostras (holdout)')

def contar_classes(ds, label):
    diab = nao = 0
    for ex in ds:
        if 'Não Diabético' in ex['messages'][1]['content']:
            nao += 1
        else:
            diab += 1
    tot = diab + nao
    print(f'  {label}: Diabético {diab} ({100*diab/tot:.0f}%) | Não Diabético {nao} ({100*nao/tot:.0f}%)')

contar_classes(dataset['train'],      'Treino')
contar_classes(dataset['validation'], 'Validação')
contar_classes(test_dataset_raw,      'Teste')


## 5. Carregar Modelo Base

> **Colab T4 (15 GB VRAM):** `USE_QLORA = True` é obrigatório.  
> **Colab A100 (40/80 GB):** `USE_QLORA = False` para maior velocidade.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ⚙️ Mude para False se estiver numa A100 / L4
USE_QLORA = True

_hf_kwargs = dict(token=HF_TOKEN or None, trust_remote_code=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **_hf_kwargs)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'

if USE_QLORA:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config,
        device_map='auto', torch_dtype=torch.bfloat16, **_hf_kwargs,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, device_map='auto', torch_dtype=torch.bfloat16, **_hf_kwargs,
    )

model.config.use_cache = False
print(f'✅ Modelo carregado: {MODEL_ID}')
print(f'   VRAM alocada    : {torch.cuda.memory_allocated() / 1e9:.2f} GB')
print(f'   Modo            : {"QLoRA 4-bit" if USE_QLORA else "bf16 puro"}')


## 6. Configurar LoRA

In [ ]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

if USE_QLORA:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.10,
    bias='none',
    task_type='CAUSAL_LM',
    use_rslora=True,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 7. Formatar Dataset no Chat Template

In [ ]:
def format_chat(example):
    return {'text': tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False
    )}

# num_proc=1 — evita deadlock de fork com o tokenizer no Colab
train_dataset = dataset['train'].map(format_chat,       num_proc=1, desc='Treino')
val_dataset   = dataset['validation'].map(format_chat,  num_proc=1, desc='Validação')
test_dataset  = test_dataset_raw.map(format_chat,       num_proc=1, desc='Teste')

sample_lengths = [len(tokenizer(t)['input_ids']) for t in train_dataset['text'][:200]]
p95_idx = int(len(sample_lengths) * 0.95)
print(f'Comprimento de tokens (amostra 200):')
print(f'  Mediana: {statistics.median(sample_lengths):.0f}')
print(f'  P95    : {sorted(sample_lengths)[p95_idx]:.0f}')
print(f'  Máximo : {max(sample_lengths)}')
print('\nExemplo (primeiros 500 chars):')
print(train_dataset[0]['text'][:500])


## 8. Callback de Validação Numérica

A cada epoch, gera respostas para 30 amostras e mede acerto de porcentagem exata e label.


In [ ]:
from transformers import TrainerCallback

class ValidacaoNumericaCallback(TrainerCallback):
    def __init__(self, model, tokenizer, val_jsonl: str, n_amostras: int = 30):
        self._model     = model
        self._tokenizer = tokenizer
        self.amostras   = []
        with open(val_jsonl) as f:
            for i, line in enumerate(f):
                if i >= n_amostras: break
                self.amostras.append(json.loads(line))

    def on_epoch_end(self, args, state, control, **kwargs):
        m  = self._model
        tk = self._tokenizer
        m.eval()
        acertos_pct = acertos_label = 0

        for obj in self.amostras:
            user   = obj['messages'][1]['content']
            pred_m = re.search(r'\[CLASSIFICAÇÃO\]\s*([^\n]+)', user)
            prob_m = re.search(r'\[PROBABILIDADE\]\s*(\d+)%', user)
            if not pred_m or not prob_m:
                continue

            pred_real = pred_m.group(1).strip()
            pct_real  = prob_m.group(1)

            prompt = tk.apply_chat_template(
                obj['messages'][:-1], tokenize=False, add_generation_prompt=True
            )
            inputs = tk(prompt, return_tensors='pt').to(m.device)

            with torch.no_grad():
                out = m.generate(
                    **inputs, max_new_tokens=150, temperature=0.1,
                    do_sample=True, pad_token_id=tk.eos_token_id,
                )

            resp = tk.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

            if pct_real + '%' in resp:            acertos_pct   += 1
            if pred_real.lower() in resp.lower(): acertos_label += 1

        n  = len(self.amostras)
        ps = 100 * acertos_pct   / n if n else 0
        ls = 100 * acertos_label / n if n else 0

        print(f'\n📊 [Epoch {state.epoch:.0f}] % exato: {acertos_pct}/{n} ({ps:.0f}%) | '
              f'Label: {acertos_label}/{n} ({ls:.0f}%)')

        if MLFLOW_OK and mlflow.active_run():
            mlflow.log_metrics(
                {'val_pct_accuracy': ps, 'val_label_accuracy': ls},
                step=int(state.epoch)
            )

        m.train()  # obrigatório — volta ao modo treino

print('✅ Callback definido.')


## 9. Configurar o Treinamento

> **Estimativa Colab T4:** ~2–3 horas para 3 épocas com QLoRA.  
> **Colab A100:** ~60–90 min em bf16 puro.


In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

training_args = SFTConfig(
    output_dir=LOCAL_OUT,

    # Épocas e batch
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,

    # Otimizador
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.10,
    optim='adamw_torch_fused',
    weight_decay=0.01,
    max_grad_norm=1.0,

    # Precisão
    fp16=not USE_QLORA,
    bf16=not USE_QLORA and torch.cuda.is_bf16_supported(),
    gradient_checkpointing=USE_QLORA,

    # Logging e eval
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,

    # Sequência
    max_seq_length=1024,
    dataset_text_field='text',
    dataset_num_proc=1,

    # Reproducibilidade
    seed=42,
    data_seed=42,

    # MLflow
    report_to='mlflow' if MLFLOW_OK else 'none',
    run_name='llama3.2_diabetes_lora_v5',
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    callbacks=[
        ValidacaoNumericaCallback(model, tokenizer, VAL_JSONL, n_amostras=30),
        EarlyStoppingCallback(early_stopping_patience=5),
    ],
)

steps_per_epoch = len(train_dataset) // (
    training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
)
total_steps = steps_per_epoch * training_args.num_train_epochs
print(f'Steps/época : {steps_per_epoch} | Total: {total_steps}')
print(f'Estimativa  : {total_steps / 12:.0f} min')


## 10. Treinar

> **Correção aplicada:** `evaluate()` é chamado **antes** de `save_model` para evitar
> `RuntimeError: on_train_begin must be called before on_evaluate` do NotebookProgressCallback.


In [ ]:
# Garante que não há run ativa orphã de execuções anteriores
if MLFLOW_OK:
    mlflow.end_run()

mlflow_run = mlflow.start_run() if MLFLOW_OK else None

try:
    if MLFLOW_OK:
        safe = {k: str(v) for k, v in training_args.to_sanitized_dict().items()
                if isinstance(v, (str, int, float, bool))}
        mlflow.log_params(safe)

    trainer_stats = trainer.train()

    # ✅ evaluate() ANTES de save_model — enquanto o estado do trainer ainda está ativo
    final_metrics = trainer.evaluate()
    print('\n=== Métricas finais ===')
    for k, v in final_metrics.items():
        print(f'  {k}: {v:.4f}')

    # Salva adapter + tokenizer
    trainer.save_model(LOCAL_OUT)
    tokenizer.save_pretrained(LOCAL_OUT)

    if MLFLOW_OK:
        mlflow.log_metrics({f'final_{k}': v for k, v in final_metrics.items()})
        mlflow.log_artifact(LOCAL_OUT, artifact_path='lora_adapter')

    print(f'\n✅ Treinamento concluído! Modelo salvo em: {LOCAL_OUT}')

finally:
    if mlflow_run:
        mlflow.end_run()


## 11. Registrar Adapter no MLflow (opcional)

In [ ]:
# ✅ Garante que não há run ativa antes de abrir uma nova
mlflow.end_run()

if not MLFLOW_OK:
    print('⚠️ MLflow indisponível — pulando registro.')
else:
    with mlflow.start_run(run_name='llama3.2_diabetes_v5_registro') as run:
        mlflow.log_artifact(LOCAL_OUT, artifact_path='lora_adapter_v5')
        mlflow.set_tags({
            'model_base':        MODEL_ID,
            'finetuning_method': 'LoRA r=32 RSLoRA',
            'dataset_version':   'v5',
        })
        print(f'✅ Adapter registrado no MLflow. run_id: {run.info.run_id}')


## 12. Carregar Modelo para Inferência

> **Correção aplicada:** `model_inf` é sempre recarregado do disco via `PeftModel`,
> garantindo que testamos o adapter salvo — não o objeto em memória do treino.


In [ ]:
from peft import PeftModel

# ✅ Sempre recarrega do disco — independente do estado da sessão
print('Carregando modelo base + adapter para inferência...')
_hf_kwargs = dict(token=HF_TOKEN or None, trust_remote_code=True)

base_model_inf = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map='auto',
    torch_dtype=torch.bfloat16 if not USE_QLORA else torch.float16,
    **_hf_kwargs,
)
model_inf = PeftModel.from_pretrained(base_model_inf, LOCAL_OUT)
model_inf.eval()
print(f'✅ model_inf carregado de: {LOCAL_OUT}')

# ---- System prompt ----
SYSTEM_PROMPT = """Você é um assistente de apoio à decisão clínica especializado em endocrinologia.

REGRAS DE INTERPRETAÇÃO:
- Probabilidade < 0.3 (30%): BAIXO risco de diabetes
- Probabilidade 0.3-0.7 (30-70%): Risco MODERADO, requer monitoramento
- Probabilidade > 0.7 (70%): ALTO risco de diabetes

IMPORTANTE:
- Reporte a probabilidade EXATAMENTE como fornecida no campo [PROBABILIDADE]
- NUNCA inverta o valor da probabilidade
- NUNCA classifique prob < 0.3 como alto risco
- Glucose < 100 mg/dL = normal | 100-125 = pré-diabetes | >= 126 = diabetes

Nunca forneça diagnóstico final; apresente como 'probabilidade sugerida pelo modelo'."""


def gerar_laudo(patient: dict, ml_out: dict, max_new_tokens: int = 400) -> str:
    prob    = ml_out['probability']
    pred    = ml_out['prediction']
    fatores = ml_out.get('main_factors', [])
    pct     = round(prob * 100)
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': (
            f'Interprete o diagnóstico de diabetes gerado pelo modelo de ML.\n\n'
            f'[CLASSIFICAÇÃO] {pred}\n'
            f'[PROBABILIDADE] {pct}% (valor exato: {prob})\n'
            f'[FATORES PRINCIPAIS] {", ".join(fatores)}\n\n'
            f'Dados do paciente: {json.dumps(patient, ensure_ascii=False)}\n\n'
            f'Saída do modelo de ML: {json.dumps(ml_out, ensure_ascii=False)}'
        )},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model_inf.device)
    with torch.no_grad():
        out = model_inf.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.1, top_p=0.9, do_sample=True,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print('✅ Função gerar_laudo definida.')


## 13. Teste Manual

In [ ]:
patient_teste = {
    'Pregnancies': 2, 'Glucose': 180, 'BloodPressure': 85,
    'SkinThickness': 28, 'Insulin': 0, 'BMI': 36.2,
    'DiabetesPedigreeFunction': 0.834, 'Age': 55,
}
ml_out_teste = {
    'prediction': 'Diabético', 'probability': 0.92,
    'main_factors': ['Glucose', 'BMI', 'Age']
}

response = gerar_laudo(patient_teste, ml_out_teste)
pct_esp  = round(ml_out_teste['probability'] * 100)

print('=' * 60)
print('RESPOSTA DO MODELO:')
print('=' * 60)
print(response)
print('\n--- Verificação ---')
print(f'{"✅" if str(pct_esp)+"%" in response else "❌"} Porcentagem {pct_esp}% presente')
print(f'{"✅" if ml_out_teste["prediction"].lower() in response.lower() else "❌"} Label presente')


## 14. Validação em Lote — Detecção de Viés

In [ ]:
test_cases = [
    {'glucose': 180, 'bmi': 36.2, 'age': 55,  'label': 'Diabético',     'prob': 0.92, 'focus': 'Alto risco'},
    {'glucose': 140, 'bmi': 28.5, 'age': 45,  'label': 'Diabético',     'prob': 0.65, 'focus': 'Risco moderado'},
    {'glucose': 200, 'bmi': 32.1, 'age': 60,  'label': 'Diabético',     'prob': 0.98, 'focus': 'Hiperglicemia severa'},
    {'glucose': 90,  'bmi': 22.1, 'age': 25,  'label': 'Não Diabético', 'prob': 0.05, 'focus': '⚠️ VIÉS: prob=5%'},
    {'glucose': 165, 'bmi': 39.0, 'age': 50,  'label': 'Diabético',     'prob': 0.88, 'focus': 'Obesidade III'},
    {'glucose': 115, 'bmi': 27.2, 'age': 40,  'label': 'Não Diabético', 'prob': 0.35, 'focus': '⚠️ VIÉS: moderado'},
    {'glucose': 190, 'bmi': 30.5, 'age': 35,  'label': 'Diabético',     'prob': 0.91, 'focus': 'Alto risco jovem'},
    {'glucose': 125, 'bmi': 26.0, 'age': 58,  'label': 'Diabético',     'prob': 0.72, 'focus': 'Limítrofe alto'},
    {'glucose': 85,  'bmi': 21.0, 'age': 22,  'label': 'Não Diabético', 'prob': 0.02, 'focus': '⚠️ VIÉS: prob=2%'},
    {'glucose': 98,  'bmi': 23.5, 'age': 30,  'label': 'Não Diabético', 'prob': 0.08, 'focus': '⚠️ VIÉS: jovem saudável'},
    {'glucose': 72,  'bmi': 19.8, 'age': 21,  'label': 'Não Diabético', 'prob': 0.03, 'focus': '⚠️ VIÉS: prob=3%'},
    {'glucose': 155, 'bmi': 41.5, 'age': 62,  'label': 'Diabético',     'prob': 0.95, 'focus': 'Muito alto risco'},
]

bias_failures = []
pct_failures  = []

for i, c in enumerate(test_cases):
    prob = c['prob']; pred = c['label']; pct = round(prob * 100)
    patient = {'Pregnancies': 1, 'Glucose': c['glucose'], 'BloodPressure': 70,
               'SkinThickness': 20, 'Insulin': 79, 'BMI': c['bmi'],
               'DiabetesPedigreeFunction': 0.5, 'Age': c['age']}
    ml_out  = {'prediction': pred, 'probability': prob, 'main_factors': ['Glucose', 'BMI', 'Age']}

    resp     = gerar_laudo(patient, ml_out, max_new_tokens=250)
    resp_low = resp.lower()

    has_bias = (prob < 0.3) and (
        'alto risco' in resp_low or 'probabilidade alta' in resp_low or
        ('diabético' in resp_low and 'não diabético' not in resp_low and 'baixo' not in resp_low)
    )
    pct_wrong = f'{pct}%' not in resp

    if has_bias:  bias_failures.append(i + 1)
    if pct_wrong: pct_failures.append(i + 1)

    status = '❌ VIÉS' if has_bias else ('⚠️ %?' if pct_wrong else '✅')
    print(f'#{i+1:2d} {status}  {c["focus"]}  | esperado: {pred} ({pct}%)')
    print(f'     {resp[:160]}...')
    print()

print(f'VIÉS     : {len(bias_failures)}/{len(test_cases)} — {bias_failures or "Nenhum ✅"}')
print(f'% ERRADA : {len(pct_failures)}/{len(test_cases)} — {pct_failures or "Nenhuma ✅"}')


## 15. Avaliação Final no Holdout

> **Correção aplicada:** usa `model_inf` (carregado do disco na célula 12), não o `model` do treino.


In [ ]:
# ✅ model_inf foi definido na célula 12 — recarregue-a se reiniciou a sessão

acertos_pct = acertos_label = total_test = 0
N_TEST = 50

with open(TEST_JSONL) as f:
    for i, line in enumerate(f):
        if i >= N_TEST: break
        obj  = json.loads(line)
        user = obj['messages'][1]['content']

        pred_m = re.search(r'\[CLASSIFICAÇÃO\]\s*([^\n]+)', user)
        prob_m = re.search(r'\[PROBABILIDADE\]\s*(\d+)%', user)
        if not pred_m or not prob_m: continue

        pred_real = pred_m.group(1).strip()
        pct_real  = prob_m.group(1)
        total_test += 1

        prompt = tokenizer.apply_chat_template(
            obj['messages'][:-1], tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors='pt').to(model_inf.device)

        with torch.no_grad():
            out = model_inf.generate(
                **inputs, max_new_tokens=250, temperature=0.1,
                do_sample=True, pad_token_id=tokenizer.eos_token_id,
            )

        resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

        if pct_real + '%' in resp:            acertos_pct   += 1
        if pred_real.lower() in resp.lower(): acertos_label += 1

pct_acc   = 100 * acertos_pct   / total_test if total_test else 0
label_acc = 100 * acertos_label / total_test if total_test else 0

print(f'=== HOLDOUT ({total_test} amostras) ===')
print(f'% exato : {acertos_pct}/{total_test} ({pct_acc:.1f}%)')
print(f'Label   : {acertos_label}/{total_test} ({label_acc:.1f}%)')

# ✅ mlflow.end_run() garante que não há run ativa orphã
if MLFLOW_OK:
    mlflow.end_run()
    with mlflow.start_run(run_name='v5_holdout_eval'):
        mlflow.log_metrics({'test_pct_accuracy': pct_acc, 'test_label_accuracy': label_acc})
    print('✅ Métricas do holdout salvas no MLflow.')


---
# Relatório — Fine-Tuning LLaMA 3.2 3B v5

## Dataset v5

| Arquivo | Exemplos | Diabético | Não Diabético | Bugs |
|---|---|---|---|---|
| `train_v5.jsonl` | 3.000 | 64% | 36% | **0** |
| `val_v5.jsonl`   | 600   | 62% | 38% | **0** |
| `test_v5.jsonl`  | 300   | 67% | 33% | **0** |

## Stack técnico
- **Modelo base:** `meta-llama/Llama-3.2-3B-Instruct`
- **LoRA:** r=32, alpha=64, RSLoRA, dropout=0.10
- **Treino:** QLoRA 4-bit (T4) ou bf16 puro (A100), batch efetivo 32, cosine LR, early stopping patience=5
- **Inferência:** temperature=0.1, anchors `[CLASSIFICAÇÃO]` e `[PROBABILIDADE]` no prompt
- **Monitoramento:** `ValidacaoNumericaCallback` + MLflow

## Bugs corrigidos (v4 → v5 → Colab)

| # | Problema | Correção |
|---|---|---|
| 1 | `trainer.evaluate()` após `save_model` — RuntimeError | Movido para antes do save |
| 2 | `mlflow.start_run()` com run ativa — Exception | `mlflow.end_run()` antes de cada `start_run` |
| 3 | `model_inf` não definido após reinício de sessão | Célula 12 recarrega do disco via `PeftModel` |
| 4 | Caminhos RunPod `/workspace/` | Adaptado para `/content/drive/MyDrive/` |
| 5 | HF Token hardcoded | `userdata.get('HF_TOKEN')` (Colab Secrets) |
| 6 | `USE_QLORA=False` inadequado para T4 | Default `True`, flag clara para A100 |
